<a href="https://colab.research.google.com/github/rushibedre/B2B-Supply-Chain-Command-Center/blob/main/New_Rule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import io
import os
import pandas as pd
import numpy as np

import gspread
from gspread_dataframe import set_with_dataframe

from google.colab import auth
auth.authenticate_user()

import gspread
from gspread_dataframe import set_with_dataframe
from google.auth import default

from google.oauth2 import service_account


creds, _ = default()
gc = gspread.authorize(creds)

sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('New Rule')


sheet = gc.open_by_key("1MUtDkbiCmuYY2xZq71icPuwUd156oA75rrEWa-gkgXY")
ws1 = sheet.worksheet('New Rule')


#reading and making consolidated Amazon reports
az_ish = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/Az+Cocoblu_Ishin_RAW.csv')
az_ish['Brand'] = az_ish['lbrBrandName']
az_anb = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/Az+Cocoblu_Anubhutee_RAW.csv')
az_anb['Brand'] = az_anb['lbrBrandName']
az_df = pd.concat([az_ish, az_anb], ignore_index=True)

#reading increff data
i_df = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/INCREFF_RAW.csv')

#modifying increff data
i_df['Date'] = pd.to_datetime(i_df['System Order Time'])
i_df['Date'] = i_df['Date'].dt.date
i_df['Day'] = i_df['Date'].apply(lambda x: x.day)
i_df['Month'] = i_df['Date'].apply(lambda x: x.month)
i_df['Year'] = i_df['Date'].apply(lambda x: x.year)

i_df = i_df[~i_df['Sales Channel'].isin(["INTERNAL", "MENSA_ERP", "COCOBLU"])]

mapping = {
    "MYNTRAV4":"Myntra",
    "FLIPKARTV3":"Flipkart",
    "AJIO_VMS":	"Ajio",
    "TATACLIQ":	"Tata Cliq",
    "SNAPDEAL_BLR":	"Snapdeal",
    "MENSA_MEESHO_MUM":	"Meesho",
    "SNAPDEAL_MUM":	"Snapdeal",
    "SNAPDEAL":	"Snapdeal",
    "MENSA_MEESHO_GGN":	"Meesho",
    "NYKAA_FASHION":	"Nykaa",
    "LIMEROAD_MENSA_GURGAON":	"Limeroad",
    "SHOPIFY_ISHIN":	"Shopify",
    "SHOPIFY_ANUBHUTEE":	"Shopify",
    "LIMEROAD_MENSA_BHIWANDI":	"Limeroad",
    "LIMEROAD_MENSA_BLR":	"Limeroad",
    "AMAZON_SC":"Az SC"}


i_df['Sales Channel'] = i_df['Sales Channel'].replace(mapping)
i_df['Sales Qty'] = i_df['Order Qty'] - i_df['Cancelled Qty']

#cleaning and consolidating increff data
idf_final = i_df[['Brand','Sales Channel', 'Date', 'Day', 'Month', 'Year', 'Style',  'Order Qty', 'Cancelled Qty', 'Sales Qty', 'Order Amount']].rename(columns={'Order Amount': 'GMV'})
increff_sales = idf_final.groupby([ 'Brand','Sales Channel','Date', 'Day', 'Month', 'Year','Style'])[['Order Qty','Cancelled Qty' ,'Sales Qty','GMV',]].sum().reset_index()
increff_sales['ASP'] = increff_sales['GMV'] / increff_sales['Order Qty']

#mapping and modifying amazon data
az_ISHmp = pd.read_excel('/content/drive/MyDrive/Setup2.0/Source/AZ Ishin+Anubbhutee Mapping.xlsx',sheet_name='ISHIN')
az_ANBmp = pd.read_excel('/content/drive/MyDrive/Setup2.0/Source/AZ Ishin+Anubbhutee Mapping.xlsx',sheet_name='ANUBHUTEE')
az_mp = pd.concat([az_ISHmp, az_ANBmp], ignore_index=True)


az_df['Sales Channel'] = 'Cocoblu'
az_df['Date'] = pd.to_datetime(az_df.apply(lambda row: f"{row['orderYear']}-{row['orderMonth']}-{row['orderDay']}", axis=1))
az_df['Date'] = az_df['Date'].dt.date


az_df['Style'] = az_df['asin'].map(az_mp.set_index('ASIN')['STYLE ID'])
az_df['Day'] = az_df['orderDay']
az_df['Month'] = az_df['orderMonth']
az_df['Year'] = az_df['orderYear']
az_df['Order Qty'] = az_df['orderQuantity']
az_df['Cancelled Qty'] = 0
az_df['Sales Qty'] = az_df['Order Qty'] - az_df['Cancelled Qty']
az_df['GMV'] = az_df['orderAmt']
az_df['ASP'] = az_df['GMV'] / az_df['Order Qty']

#cleaning and consolidating Amazon data
az_main = az_df[['Brand','Sales Channel', 'Date', 'Day', 'Month', 'Year', 'Style',  'Order Qty', 'Cancelled Qty', 'Sales Qty', 'GMV']]
az_main = az_main.groupby([ 'Brand','Sales Channel','Date', 'Day', 'Month', 'Year','Style'])[['Order Qty','Cancelled Qty' ,'Sales Qty','GMV']].sum().reset_index()
az_main['ASP'] = az_main['GMV'] / az_main['Order Qty']

#reading and modifying myntra vpl data
v_df = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/VPL_RAW.csv')
v_df['Date'] = pd.to_datetime(v_df['Order Date as dd/mm/yyyy hh:MM:ss'])
v_df['Date'] = v_df['Date'].dt.date
v_df['Day'] = v_df['Date'].apply(lambda x: x.day)
v_df['Month'] = v_df['Date'].apply(lambda x: x.month)
v_df['Year'] = v_df['Date'].apply(lambda x: x.year)

v_df['Sales Channel'] = 'VPL'
# Using str.split() and fillna() for robustness instead of apply
v_df['temp_brand'] = v_df['Item Type Name'].str.split().str[0]
v_df['Brand'] = v_df['Item Type Brand'].fillna(v_df['temp_brand'])
v_df = v_df.drop(columns=['temp_brand'])

mapping = {
    "Ishin":"Ishin",
    "Anubhutee":"Anubhutee",
    "shin":"Ishin"
}
v_df['Brand'] = v_df['Brand'].replace(mapping)
v_df = v_df[v_df['Brand'].isin(["Ishin", "Anubhutee"])]
v_df['Order Qty'] = 1
cancel_conditions = [
    (v_df['Sale Order Status']=='COMPLETE'),
    (v_df['Sale Order Status']=='PROCESSING'),
    (v_df['Sale Order Status']=='CANCELLED')
]
cancel_choices = [0, 0, 1]
v_df['Cancelled Qty'] = np.select(cancel_conditions, cancel_choices, default=0)
v_df['Sales Qty'] = v_df['Order Qty'] - v_df['Cancelled Qty']

v_df['Style'] = v_df['Item SKU Code']
v_df['GMV'] = v_df['Selling Price']*v_df['Sales Qty']


#cleaning and consolidating vpl data
v_df = v_df[['Brand','Sales Channel', 'Date', 'Day', 'Month', 'Year', 'Style',  'Order Qty', 'Cancelled Qty', 'Sales Qty', 'GMV']]
vpl_sales = v_df.groupby([ 'Brand','Sales Channel','Date', 'Day', 'Month', 'Year','Style'])[['Order Qty','Cancelled Qty' ,'Sales Qty','GMV',]].sum().reset_index()
vpl_sales['ASP'] = vpl_sales['GMV'] / vpl_sales['Sales Qty']


#combining all dataframes
combined_df = pd.concat([increff_sales, az_main, vpl_sales], ignore_index=True)
combined_df['Prod Val'] = combined_df['Sales Qty']*combined_df['ASP']

ws.batch_clear(['A1:M'])
ws1.batch_clear(['A1:M'])

set_with_dataframe(ws, combined_df)
set_with_dataframe(ws1, combined_df)

### ME_Cross_border

In [ ]:
# Read ME sales data from Excel file
me_df = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/ME_Dump.csv')

In [ ]:
filtered_df = me_df[(me_df['brand_en'] == 'ISHIN') & (me_df['item_status'] != 'cancelled')]
# 'Excluding Tax' (5% tax and 15%)
filtered_df['Excluding Tax'] = np.where(filtered_df['country_code'] == 'ae', filtered_df['target_base_price'] / (1+0.05), np.where(filtered_df['country_code'] == 'sa',filtered_df['target_base_price'] / (1+0.15) ,np.nan))

# 'Returns' (30% return rate)
filtered_df['After Returns'] = filtered_df['Excluding Tax']

# NR in INR
filtered_df['NR in INR'] = filtered_df['After Returns'] * 23

#GMV
filtered_df['GMV'] = filtered_df['target_base_price'] * 23

# Authorize and access the sheet (you already have this part)
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('ME Dump')

# Write DataFrame to the worksheet
ws.batch_clear(['A1:AJ'])
set_with_dataframe(ws, filtered_df)


### Stylii

In [ ]:
# Read ME sales data from Excel file
stylii_df = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/Stylii_Dump.csv')

In [ ]:
# Convert date format
stylii_df['Order Date'] = pd.to_datetime(stylii_df['Order Date'],format='mixed',dayfirst=True,errors='coerce')
stylii_df['Date'] = stylii_df['Order Date'].dt.date

# NR in AED
stylii_df['NR in AED'] = stylii_df['Net Retail Value'] * 0.75 * 0.85 * 0.58

# NR in INR
stylii_df['NR in INR'] = stylii_df['NR in AED'] * 23

# Authorize and access the sheet (you already have this part)
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('Stylii')

# Write DataFrame to the worksheet
ws.batch_clear(["A1:J1000"])
set_with_dataframe(ws, stylii_df)

### Noon_CB

In [ ]:
# Read ME sales data from Excel file
noon_df = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/Noon_Dump.csv')

In [ ]:
filtered_df = noon_df[(noon_df['brand_en'] == 'ISHIN') & (noon_df['item_status'] == 'Delivered')]
# 'Excluding Tax' (5% tax and 15%)
filtered_df['Excluding Tax'] = np.where(filtered_df['country_code'] == 'AE', filtered_df['base_price'] / (1+0.05), np.where(filtered_df['country_code'] == 'SA',filtered_df['base_price'] / (1+0.15) ,np.nan))

# 'Returns' (30% return rate)
filtered_df['After Returns'] = filtered_df['Excluding Tax'] * 0.70

# NR in INR
filtered_df['NR in INR'] = filtered_df['After Returns'] * 23

#GMV
filtered_df['GMV'] = filtered_df['base_price'] * 23

# Authorize and access the sheet (you already have this part)
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('Noon Dump')

# Write DataFrame to the worksheet
ws.batch_clear(['A1:AZ'])
set_with_dataframe(ws, filtered_df)


### namshi_fbn

In [ ]:
# Read ME sales data from Excel file
namshi_fbn = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/namshi_fbn_dump.csv')

In [ ]:
filtered_df = namshi_fbn[(namshi_fbn['brand_en'] == 'ISHIN')]
# 'Excluding Tax' (5% tax and 15%)
tax_rates = {'ae': 0.05, 'sa': 0.15, 'bh': 0.05, 'kw': 0.05}

filtered_df['Excluding Tax'] = filtered_df.apply(
    lambda x: x['target_base_price'] / (1 + tax_rates.get(x['country_code'], np.nan))
    if x['country_code'] in tax_rates else np.nan,
    axis=1)

# 'Returns' (35% return rate)
filtered_df['After Returns'] = filtered_df['Excluding Tax'] * 0.65

# NR in INR
filtered_df['NR in INR'] = filtered_df['After Returns'] * 23

#GMV
filtered_df['GMV'] = filtered_df['target_base_price'] * 23

# Authorize and access the sheet (you already have this part)
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('namshi_fbn')

# Write DataFrame to the worksheet
ws.batch_clear(['A1:AZ'])
set_with_dataframe(ws, filtered_df)


### Noon FBN

In [ ]:
# Read ME sales data from Excel file
noon_fbn = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/noon_fbn_dump.csv')

In [ ]:
filtered_df = noon_fbn[(noon_fbn['brand_en'] == 'ISHIN') & (noon_fbn['item_status'] == 'Delivered')]
# 'Excluding Tax' (5% tax and 15%)
tax_rates = {'AE': 0.05, 'SA': 0.15, 'BH': 0.05, 'KW': 0.05}

filtered_df['Excluding Tax'] = filtered_df.apply(
    lambda x: x['base_price'] / (1 + tax_rates.get(x['country_code'], np.nan))
    if x['country_code'] in tax_rates else np.nan,
    axis=1)

# 'Returns' (30% return rate)
filtered_df['After Returns'] = filtered_df['Excluding Tax']

# NR in INR
filtered_df['NR in INR'] = filtered_df['After Returns'] * 23

#GMV
filtered_df['GMV'] = filtered_df['base_price'] * 23

# Authorize and access the sheet
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('noon_fbn')

# Write DataFrame to the worksheet
ws.batch_clear(['A1:AZ'])
set_with_dataframe(ws, filtered_df)


### Noon B2B Sales

In [ ]:
# Read ME sales data from Excel file
noon_b2b_ae = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/noon_ae_b2b.csv')
noon_b2b_sa = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/noon_sa_b2b.csv')

noon_b2b_combined = pd.concat([noon_b2b_ae, noon_b2b_sa], ignore_index=True)

noon_b2b_combined = noon_b2b_combined[ (noon_b2b_combined['brand_code'] == 'ishin')]

noon_b2b_combined['order_timestamp'] = pd.to_datetime(noon_b2b_combined['order_timestamp'])
noon_b2b_combined['order_date'] = noon_b2b_combined['order_timestamp'].dt.date

In [ ]:
# Authorize and access the sheet
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('noon_b2b')

# Write DataFrame to the worksheet
ws.batch_clear(['A2:Q'])
set_with_dataframe(ws, noon_b2b_combined, row=2, col=1, include_column_header=True)

### Noon FBN


In [ ]:
# Read ME sales data from Excel file
noon_fbn_ae = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/noon_ae_fbn.csv')
noon_fbn_sa = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/noon_sa_fbn.csv')

noon_fbn_combined = pd.concat([noon_fbn_ae, noon_fbn_sa], ignore_index=True)

noon_fbn_combined = noon_fbn_combined[ (noon_fbn_combined['brand_code'] == 'ishin') ]

noon_fbn_combined['order_timestamp'] = pd.to_datetime(noon_fbn_combined['order_timestamp'])
noon_fbn_combined['order_date'] = noon_fbn_combined['order_timestamp'].dt.date

In [ ]:
# Authorize and access the sheet
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('noon_fbn_')

# Write DataFrame to the worksheet
ws.batch_clear(['A2:Q'])
set_with_dataframe(ws, noon_fbn_combined, row=2, col=1, include_column_header=True)

### Cocoblu Secondary Sales

In [ ]:
# Read Cocoblu sales data from Excel file
cocoblu_ishin = pd.read_csv('/content/drive/MyDrive/Setup2.0/Source/cocoblu_ishin_Secondary_sales.csv')

In [ ]:
cocoblu_ishin.columns

In [ ]:
cocoblu_ishin.head()

In [ ]:
# Authorize and access the sheet
sheet = gc.open_by_key("1eD8wTJXQVH7jEHvgHNfsoMikO1FQltu5t63M1N9bWAQ")
ws = sheet.worksheet('cocoblu_ishin')

# Write DataFrame to the worksheet
ws.batch_clear(['A2:P'])
set_with_dataframe(ws, cocoblu_ishin, row=2, col=1, include_column_header=True)